# Lesson 07 Lab — Filter Pruning: Making Convolution Physically Narrower

**Puzzle:** Why does zeroing filters differ from deleting them?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

A dense convolution library receives input/output channel counts, kernel size, stride, and dtype. Setting complete filters to zero preserves those dimensions. Physical filter pruning constructs a smaller convolution and propagates the selected channels to the next layer, allowing ordinary dense kernels to execute less work.


## 0. Predict before running

1. Predict the output shapes of masked and physically pruned blocks.
2. Predict which candidate reduces analytical convolution work.
3. Identify the exact slice that must be applied to the second convolution.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

Two consecutive convolutions form the concrete dependency. One candidate masks half of the first layer's output filters; another physically copies the retained filters and the matching input-channel slices of the second convolution.

- Filter zeros preserve the dense convolution descriptor.
- Physical pruning changes two coupled channel dimensions.
- Equivalence should be checked before a latency claim.


## 2. Derive the mechanism

For a convolution, leading work scales with `N × Hout × Wout × Cout × Cin × Kh × Kw`. A zeroed filter leaves Cout unchanged in the operator descriptor. Deleting it halves the first layer's Cout and the next layer's Cin when dependencies are propagated. Copying the same retained weights provides an equivalence check between masked and narrowed functions before timing.

Keep value sparsity, physical shape, representation, and runtime evidence separate.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 7
LESSON_TITLE = 'Filter Pruning: Making Convolution Physically Narrower'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260815
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | same-shape filter-masked two-convolution block |
| Candidate | physically narrowed block with propagated second-layer input channels |
| Held constant | input tensor, retained filter indices, copied weights, batch, spatial shape, dtype, and timing |
| Measurements | output max error, parameters, analytical FLOPs, median latency, and channel shapes |
| Evidence | `pytorch-gpu` |

**Experiment:** Mask and physically remove the same convolution filters, then compare equivalence, parameters, FLOPs, and CUDA latency.


## 5. Read the experiment code

The physical model is rebuilt with smaller module dimensions and receives exact weight slices from the masked model. Summing the retained first-layer channels into the second layer is not approximated: the matching input-channel axis is sliced. Near-zero output drift verifies the dependency before speed and parameter numbers are interpreted.

Do not execute until the code implements the frozen table above.


In [2]:
dtype = torch.bfloat16
class Block(nn.Module):
    def __init__(self, c1=32):
        super().__init__(); self.conv1 = nn.Conv2d(16, c1, 3, padding=1, bias=False, dtype=dtype); self.conv2 = nn.Conv2d(c1, 24, 3, padding=1, bias=False, dtype=dtype)
    def forward(self, x): return self.conv2(F.relu(self.conv1(x)))

full = Block().to(DEVICE).eval(); masked = copy.deepcopy(full)
keep = torch.arange(0, 32, 2, device=DEVICE); remove = torch.arange(1, 32, 2, device=DEVICE)
with torch.no_grad(): masked.conv1.weight[remove] = 0
narrow = Block(c1=keep.numel()).to(DEVICE).eval()
with torch.no_grad():
    narrow.conv1.weight.copy_(full.conv1.weight[keep])
    narrow.conv2.weight.copy_(full.conv2.weight[:, keep])
x = torch.randn(16, 16, 32, 32, device=DEVICE, dtype=dtype)
with torch.inference_mode(): masked_y, narrow_y = masked(x), narrow(x)
tm = timing_summary(cuda_times(lambda: masked(x))); tn = timing_summary(cuda_times(lambda: narrow(x)))
def conv_flops(model, batch=16, h=32, w=32):
    return int(sum(2*batch*h*w*m.out_channels*(m.in_channels//m.groups)*m.kernel_size[0]*m.kernel_size[1] for m in model.modules() if isinstance(m, nn.Conv2d)))
metrics = {
    "masked_parameters": count_params(masked), "narrow_parameters": count_params(narrow),
    "masked_flops": conv_flops(masked), "narrow_flops": conv_flops(narrow),
    "flop_reduction": 1 - conv_flops(narrow)/conv_flops(masked),
    "max_error": float((masked_y.float()-narrow_y.float()).abs().max().item()),
    "masked_median_ms": tm["median_ms"], "narrow_median_ms": tn["median_ms"],
    "masked_channels": [32,24], "narrow_channels": [int(keep.numel()),24],
}
analysis = (
    f"Masking retained {metrics['masked_parameters']:,} parameters, while physical propagation reduced the block to "
    f"{metrics['narrow_parameters']:,} and analytical convolution work by {metrics['flop_reduction']:.1%}. "
    f"The copied narrow block matched the masked control within {metrics['max_error']:.3e}. Median latency changed "
    f"from {metrics['masked_median_ms']:.6f} to {metrics['narrow_median_ms']:.6f} ms on this shape."
)


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Masked parameters | 11,520 |
| Narrow parameters | 5,760 |
| FLOP reduction | 50.00% |
| Equivalence max error | 0.001953 |
| Masked median | 0.056320 ms |
| Narrow median | 0.044656 ms |


## 7. Interpret rather than merely print

Masking retained 11,520 parameters, while physical propagation reduced the block to 5,760 and analytical convolution work by 50.0%. The copied narrow block matched the masked control within 1.953e-03. Median latency changed from 0.056320 to 0.044656 ms on this shape.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The tensors and operators executed on CUDA through PyTorch. Native sparse-kernel identity is not inferred unless a trace or backend artifact names it.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 7,
    "title": 'Filter Pruning: Making Convolution Physically Narrower',
    "environment": ENV,
    "evidence_label": 'pytorch-gpu',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'Filter pruning accelerates ordinary dense convolution only after channel dimensions are physically rebuilt and propagated.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 7,
  "title": "Filter Pruning: Making Convolution Physically Narrower",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260815
  },
  "evidence_label": "pytorch-gpu",
  "metrics": {
    "masked_parameters": 11520,
    "narrow_parameters": 5760,
    "masked_flops": 377487360,
    "narrow_flops": 188743680,
    "flop_reduction": 0.5,
    "max_error": 0.001953125,
    "masked_median_ms": 0.05632000043988228,
    "narrow_median_ms": 0.04465600103139877,
    "masked_channels": [
      32,
      24
    ],
    "narrow_channels": [
      16,
      24
    ]
  },
  "analysis": "Masking retained 11,520 parameters, while physical propagation reduced the block to 5,760 and analytical convolution work by 50.0%. The copied narrow block matched the masked control within 1.953e-03. Median latency changed from 0.056320 to 0.044656 ms on this shape.",
  "concl

## 9. Make the bounded decision

> Filter pruning accelerates ordinary dense convolution only after channel dimensions are physically rebuilt and propagated.

**Acceptance/rollback:** Accept structural filter pruning only when all consumer shapes are updated, functional drift is understood, and the target runtime improves under representative spatial sizes.

**Failure analysis:** Selecting channels independently in adjacent convolutions breaks equivalence. BatchNorm, residual adds, groups, and concatenations introduce additional dependencies not present in this two-layer probe. Awkward channel counts can also reduce kernel efficiency despite lower FLOPs.


## 10. Extend the evidence

Insert BatchNorm and a residual branch, then use a dependency graph to enumerate every coupled slice. Sweep retained widths that align and misalign with the target convolution backend.

The full evidence boundary and references are in [`README.md`](README.md).
